# DX 704 Week 10 Project

In this project, you will implement document search within a question and answer database and assess its performance.


The full project description and a template notebook are available on GitHub: [Project 10 Materials](https://github.com/bu-cds-dx704/dx704-project-10).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

In [1]:
%pip install pandas numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 24.0 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 22.5 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 32.6 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 35.1 MB/s  0:00:016m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Part 1: Download the SQuAD-explorer Data Set

You may use the code provided below.

In [2]:
!git clone https://github.com/rajpurkar/SQuAD-explorer

Cloning into 'SQuAD-explorer'...
remote: Enumerating objects: 5563, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 5563 (delta 11), reused 17 (delta 6), pack-reused 5539 (from 1)
Receiving objects: 100% (5563/5563), 52.26 MiB | 31.09 MiB/s, done.
Resolving deltas: 100% (3563/3563), done.


In [3]:
import json

In [4]:
with open("SQuAD-explorer/dataset/train-v1.1.json") as fp:
    train_data = json.load(fp)

In [5]:
type(train_data)

dict

In [6]:
list(train_data.keys())

['data', 'version']

In [7]:
type(train_data["data"])

list

In [8]:
len(train_data["data"])

442

In [9]:
type(train_data["data"][0])

dict

In [10]:
train_data["data"][0].keys()

dict_keys(['title', 'paragraphs'])

In [11]:
train_data["data"][0]["title"]

'University_of_Notre_Dame'

In [12]:
len(train_data["data"][0]["paragraphs"])

55

In [13]:
train_data["data"][0]["paragraphs"][0]

{'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'qas': [{'answers': [{'answer_start': 515,
     'text': 'Saint Bernadette Soubirous'}],
   'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
   'id': '5733be284776f41900661182'},
  {'answers': [{'answer_start': 188, 'text': 'a copper statue of Christ

In [14]:
sum(len(doc["paragraphs"]) for doc in train_data["data"])

18896

## Part 2: Restructure JSON Data for Processing

Parse the file "SQuAD-explorer/dataset/train-v1.1.json" above to produce a file "parsed.tsv" with columns document_title, paragraph_index, and paragraph_context.
The paragraph_index column should be zero-indexed, so zero for the first paragraph of each document.
Use pandas `to_csv` method to write the file since there are many quotes and other issues to handle otherwise.

In [15]:
# YOUR CHANGES HERE

import pandas as pd

rows = []
for doc in train_data["data"]:
    for i, p in enumerate(doc["paragraphs"]):
        rows.append({
            "document_title": doc["title"],
            "paragraph_index": i,
            "paragraph_context": p["context"],
        })

parsed = pd.DataFrame(rows)
parsed.to_csv("parsed.tsv", sep="\t", index=False)
parsed.head()

,document_title,paragraph_index,paragraph_context
0,University_of_Notre_Dame,0,"Architecturally, the school has a Catholic cha..."
1,University_of_Notre_Dame,1,"As at most other universities, Notre Dame's st..."
2,University_of_Notre_Dame,2,The university is the major seat of the Congre...
3,University_of_Notre_Dame,3,The College of Engineering was established in ...
4,University_of_Notre_Dame,4,All of Notre Dame's undergraduate students are...


Submit "parsed.tsv" in Gradescope.

## Part 3: Prepare Suitable Paragraph Vectors for Document Search

Design and implement paragraph vectors based on their text with length 1024.
Note that this will be much smaller than the number of distinct words in the training data.

Hint: you can base your vectors on any techniques covered in this module so far.
Beware that they will be automatically assessed (along with the question vectors of part 4) to make sure they retain useful information.

In [16]:
# YOUR CHANGES HERE

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

vectorizer = TfidfVectorizer(stop_words="english", sublinear_tf=True, min_df=2)
X_paragraphs = vectorizer.fit_transform(parsed["paragraph_context"])

svd = TruncatedSVD(n_components=1024, random_state=42)
paragraph_vectors = normalize(svd.fit_transform(X_paragraphs))
paragraph_vectors.shape

(18896, 1024)

Save your paragraph vectors in a file "paragraph-vectors.tsv.gz" with columns document_title, paragraph_index, and paragraph_vector_json where paragraph_vector_json is a JSON encoded list.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

In [17]:
# YOUR CHANGES HERE

paragraph_vectors_df = pd.DataFrame({
    "document_title": parsed["document_title"],
    "paragraph_index": parsed["paragraph_index"],
    "paragraph_vector_json": [json.dumps(v.tolist()) for v in paragraph_vectors],
})
paragraph_vectors_df.to_csv("paragraph-vectors.tsv.gz", sep="\t", index=False)

Submit "paragraph-vectors.tsv.gz" in Gradescope.

## Part 4: Encode Question Vectors with the Same Design

Read the questions in "questions.tsv" and encode them in the same way that you encoded the paragraph vectors.

In [18]:
# YOUR CHANGES HERE

questions = pd.read_csv("questions.tsv", sep="\t")
X_questions = vectorizer.transform(questions["question"])
question_vectors = normalize(svd.transform(X_questions))
question_vectors.shape

(100, 1024)

Save your question vectors in "question-vectors.tsv" with columns question_id and question_vector_json.

In [19]:
# YOUR CHANGES HERE

question_vectors_df = pd.DataFrame({
    "question_id": questions["question_id"],
    "question_vector_json": [json.dumps(v.tolist()) for v in question_vectors],
})
question_vectors_df.to_csv("question-vectors.tsv", sep="\t", index=False)

Submit "question-vectors.tsv" in Gradescope.

## Part 5: Match Questions to Paragraphs using Nearest Neighbors

Match your question vectors to paragraph vectors and identify the top 5 paragraph vectors for each question using nearest neighbors.
Specifically, use the Euclidean distance between the vectors.


In [20]:
# YOUR CHANGES HERE

from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=5, metric="euclidean")
nn.fit(paragraph_vectors)
distances, indices = nn.kneighbors(question_vectors)

Save your top matches in a file "question-matches.tsv" with columns question_id, question_rank, document_title, and paragraph_index.


In [21]:
# YOUR CHANGES HERE

match_rows = []
for qid, idx_row in zip(questions["question_id"], indices):
    for rank, idx in enumerate(idx_row, start=1):
        match_rows.append({
            "question_id": qid,
            "question_rank": rank,
            "document_title": parsed.loc[idx, "document_title"],
            "paragraph_index": parsed.loc[idx, "paragraph_index"],
        })

matches = pd.DataFrame(match_rows)
matches.to_csv("question-matches.tsv", sep="\t", index=False)
matches.head(10)

,question_id,question_rank,document_title,paragraph_index
0,1,1,Near_East,57
1,1,2,United_Nations_Population_Fund,4
2,1,3,Bill_%26_Melinda_Gates_Foundation,6
3,1,4,Boston,12
4,1,5,Association_football,18
5,4,1,BeiDou_Navigation_Satellite_System,18
6,4,2,BeiDou_Navigation_Satellite_System,12
7,4,3,BeiDou_Navigation_Satellite_System,7
8,4,4,BeiDou_Navigation_Satellite_System,14
9,4,5,BeiDou_Navigation_Satellite_System,2


Submit "question-matches.tsv" in Gradescope.

## Part 6: Spot Check Question and Paragraph Matches

Review the paragraphs matched to the first 5 questions (sorted by question_id ascending).
Which paragraph was the worst match for each question?


In [22]:
review = matches.merge(parsed, on=["document_title", "paragraph_index"]).merge(questions, on="question_id")
review = review.sort_values(["question_id", "question_rank"])

for qid in sorted(questions["question_id"])[:5]:
    sub = review[review["question_id"] == qid]
    print("=" * 100)
    print(qid, sub["question"].iloc[0])
    for _, r in sub.iterrows():
        print("--", r["question_rank"], r["document_title"], r["paragraph_index"])
        print(r["paragraph_context"][:400])
        print()

1 What was the goal of the abuse of region project?
-- 1 Near_East 57
One such institution is the Centre for the Study of Ancient Documents (CSAD) founded by and located centrally at Oxford University, Great Britain. Among its many activities CSAD numbers "a long-term project to create a library of digitised images of Greek inscriptions." These it arranges by region. The Egypt and the Near East region besides Egypt includes Cyprus, Persia and Afghanistan but not Asi

-- 2 United_Nations_Population_Fund 4
The Sustainable Development Goals are ambitious, and they will require enormous efforts across countries, continents, industries and disciplines - but they are achievable. UNFPA is working with governments, partners and other UN agencies to directly tackle many of these goals - in particular Goal 3 on health, Goal 4 on education and Goal 5 on gender equality - and contributes in a variety of ways 

-- 3 Bill_%26_Melinda_Gates_Foundation 6
The IJM used the grant money to found "Project 

Submit "worst-paragraphs.tsv" in Gradescope.

In [23]:
worst = pd.DataFrame([
    {"question_id": 1, "document_title": "Association_football", "paragraph_index": 18},
    {"question_id": 4, "document_title": "BeiDou_Navigation_Satellite_System", "paragraph_index": 2},
    {"question_id": 7, "document_title": "Beyoncé", "paragraph_index": 38},
    {"question_id": 10, "document_title": "Roman_Republic", "paragraph_index": 48},
    {"question_id": 13, "document_title": "San_Diego", "paragraph_index": 16},
])
worst.to_csv("worst-paragraphs.tsv", sep="\t", index=False)
worst

,question_id,document_title,paragraph_index
0,1,Association_football,18
1,4,BeiDou_Navigation_Satellite_System,2
2,7,Beyoncé,38
3,10,Roman_Republic,48
4,13,San_Diego,16


Write a file "worst-paragraphs.tsv" with three columns question_id, document_title, paragraph_index.

## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 8: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.